In [ ]:
import os
import requests

REPO_ID = "airvlab/Grasp-Anything"
REPO_TYPE = "datasets"
REVISION = "main"
PARTS = ["image_part_aa", "image_part_ab"]
OUT_PATH = "image.zip"

CHUNK_SIZE = 4 * 1024 * 1024  # 4 MB chunks

# Set HF_TOKEN if the dataset requires authentication / gated access
hf_token = os.environ.get("HF_TOKEN")
headers = {"Authorization": f"Bearer {hf_token}"} if hf_token else {}


def hf_resolve_url(
    repo_id: str, path_in_repo: str, revision: str = "main", repo_type: str = "datasets"
) -> str:
  prefix = "datasets/" if repo_type == "datasets" else ""
  return f"https://huggingface.co/{prefix}{repo_id}/resolve/{revision}/{path_in_repo}"


def stream_to_handle(url: str, dst_handle, headers: dict) -> int:
  written = 0
  with requests.get(
      url, stream=True, headers=headers, timeout=120, allow_redirects=True
  ) as r:
    r.raise_for_status()
    for chunk in r.iter_content(chunk_size=CHUNK_SIZE):
      if chunk:
        dst_handle.write(chunk)
        written += len(chunk)
  return written


try:
  total_bytes = 0
  with open(OUT_PATH, "wb") as out:
    for part in PARTS:
      url = hf_resolve_url(REPO_ID, part, revision=REVISION, repo_type=REPO_TYPE)
      print(f"Streaming {part} -> {OUT_PATH}...")
      part_bytes = stream_to_handle(url, out, headers)
      total_bytes += part_bytes
      print(f"Finished {part} ({part_bytes / (1024**3):.2f} GB)")

  print(
      f"Done! Combined {OUT_PATH} ({total_bytes / (1024**3):.2f} GB total)."
  )
except Exception:
  if os.path.exists(OUT_PATH):
    os.remove(OUT_PATH)
  raise

Streaming image_part_aa -> image.zip...
Finished image_part_aa (32.00 GB)
Streaming image_part_ab -> image.zip...
Finished image_part_ab (28.55 GB)
Done! Combined image.zip (60.55 GB total).


In [ ]:
#!unzip -Z1 image.zip | grep -v '/$' | sort | head -n 1 | xargs -d '\n' unzip image.zip -d .

Archive:  image.zip
  inflating: ./image/000018668931a8fb14891fa2b4c0aaa4b50334d17c20d7d2f6306cc47b2f9830.jpg  


In [ ]:
from pathlib import Path
import pickle
import re
import shutil
import zipfile

ZIP_PATH = "image.zip"
SEEN_OBJ = "seen.obj"
UNSEEN_OBJ = "unseen.obj"
OUTPUT_DIR = Path("images")

# "union"        -> extract images present in seen.obj OR unseen.obj
# "intersection" -> extract only images present in BOTH seen.obj AND unseen.obj
MATCH_MODE = "union"
CHUNK_SIZE = 1024 * 1024  # 1 MB


def load_base_hashes(obj_path: str) -> set[str]:
  """Load IDs from a pickled .obj file (or line-delimited text) and strip _<index>."""
  try:
    with open(obj_path, "rb") as f:
      data = pickle.load(f)
    items = data.keys() if isinstance(data, dict) else data
  except Exception:
    with open(obj_path, "r", encoding="utf-8") as f:
      items = [line.strip() for line in f if line.strip()]

  hashes = set()
  for item in items:
    stem = Path(str(item).strip()).stem
    # Remove trailing _0, _1, _2, etc. to get the 64-char image hash
    base_hash = re.sub(r"_\d+$", "", stem)
    if base_hash:
      hashes.add(base_hash)
  return hashes


seen_hashes = load_base_hashes(SEEN_OBJ)
unseen_hashes = load_base_hashes(UNSEEN_OBJ)

if MATCH_MODE == "intersection":
  target_hashes = seen_hashes & unseen_hashes
else:
  target_hashes = seen_hashes | unseen_hashes

print(f"Loaded {len(seen_hashes)} unique image hashes from {SEEN_OBJ}")
print(f"Loaded {len(unseen_hashes)} unique image hashes from {UNSEEN_OBJ}")
print(f"Target hashes to extract ({MATCH_MODE}): {len(target_hashes)}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
extracted_count = 0

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
  for member in zf.infolist():
    if member.is_dir():
      continue

    member_path = Path(member.filename)
    if ".." in member_path.parts or member_path.suffix.lower() != ".jpg":
      continue

    # member_path.stem is the image hash in ./image/<hash>.jpg
    if member_path.stem not in target_hashes:
      continue

    target_path = (OUTPUT_DIR / member_path.name).resolve()
    if not target_path.is_relative_to(OUTPUT_DIR.resolve()):
      continue

    with zf.open(member) as src, open(target_path, "wb") as dst:
      shutil.copyfileobj(src, dst, length=CHUNK_SIZE)

    extracted_count += 1

print(f"Extracted {extracted_count} matching images to {OUTPUT_DIR}")

Loaded 420 unique image hashes from seen(1).obj
Loaded 200 unique image hashes from unseen.obj
Target hashes to extract (union): 620
Extracted 620 matching images to extracted_matched


In [ ]:
!zip -r images.zip images

Streaming output truncated to the last 5000 lines.
  adding: extracted_matched/270eb6660e0ca81bae322c9f04a037ab600ebd068d6546b67b87fb1f5f5e2631.jpg (deflated 0%)
  adding: extracted_matched/9da06d9f723dd7c4b77824f20061fe63ce5e00c01ffac58a7c77cdf6792fcd3c.jpg (deflated 0%)
  adding: extracted_matched/4fb5f4be18b9f3ac06943132e475fba7c4cf250163e0bd1281b372da3b4ace2e.jpg (deflated 0%)
  adding: extracted_matched/75dc3072f0a2f714a5255484f8b93fee1625837a850c2bde5089b146caca5cb7.jpg (deflated 0%)
  adding: extracted_matched/5feb02b7b7f1690940039c26e044b753a5cbeb7869684a7e528292bea6eda19d.jpg (deflated 0%)
  adding: extracted_matched/92dcee3821631d5bf1367a25aa6fe41c17e20e4bed42824e836cd5e52353e8ad.jpg (deflated 0%)
  adding: extracted_matched/ea689d90d3744f089add41b9824cc5b53cc999ded644a923fda8fe0cc755101c.jpg (deflated 0%)
  adding: extracted_matched/53a8ce4f7d8894d74dd0e3c421c8ff088da5a32862f76c30b60ad317ef585d83.jpg (deflated 0%)
  adding: extracted_matched/83b2fac054ded36074494fcdf2e69425f

In [ ]:
from google.colab import files

files.download("images.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!unzip -Z1 grasp_instructions.zip | grep -v '/$' | wc -l

4412383


In [ ]:
!unzip -Z1 grasp_label_negative.zip | grep -v '/$' | wc -l

4412383


In [ ]:
!unzip -Z1 grasp_label_positive.zip | grep -v '/$' | wc -l

4412383


In [ ]:
!unzip -Z1 part_mask.zip | grep -v '/$' | wc -l

4412383
